# Bayesian Optimization for Hyperparameter Tuning

### A Complete Kaggle-Style Machine Learning Notebook

**Goal:** Understand the theory, mathematics, visualization, implementation, and comparison of Bayesian Optimization for hyperparameter tuning.

This notebook covers:

- Hyperparameters vs model parameters
- Why hyperparameter tuning is needed
- Bayesian Optimization intuition
- Mathematical formulation
- Surrogate models
- Gaussian Processes
- Acquisition functions
- Exploration vs exploitation
- End-to-end Bayesian hyperparameter optimization
- Visualization of optimization trials
- Comparison with Grid Search and Random Search
- Exam-ready summary

## 1. Learning Objectives

By the end of this notebook, you should be able to:

1. Define hyperparameter tuning.
2. Explain Bayesian Optimization in simple terms.
3. Write the mathematical optimization problem.
4. Explain the role of a surrogate model.
5. Explain Gaussian Process mean and uncertainty.
6. Explain acquisition functions such as Expected Improvement.
7. Understand exploration vs exploitation.
8. Implement Bayesian-style hyperparameter optimization with Optuna.
9. Visualize optimization trials and parameter importance.
10. Compare Grid Search, Random Search, and Bayesian Optimization.

## 2. What Are Hyperparameters?

A **hyperparameter** is a value chosen before or outside the normal model-learning process that controls how a machine learning algorithm behaves.

Examples:

| Model | Hyperparameters |
|---|---|
| Random Forest | `n_estimators`, `max_depth`, `min_samples_split` |
| SVM | `C`, `gamma`, `kernel` |
| KNN | `n_neighbors`, `weights` |
| Neural Network | `learning_rate`, `batch_size`, `dropout` |

### Model parameter vs hyperparameter

**Model parameter:** learned from the training data.

Example:
- Linear regression weights

**Hyperparameter:** selected by the practitioner/optimization algorithm.

Example:
- Random Forest tree count
- Learning rate

## 3. Why Do We Need Hyperparameter Tuning?

A model's performance depends on its hyperparameters.

For example:

```text
max_depth = 3   → accuracy = 0.82
max_depth = 5   → accuracy = 0.86
max_depth = 10  → accuracy = 0.90
max_depth = 20  → accuracy = 0.87
```

Our goal is to find the configuration that gives the best validation performance.

Mathematically:

$$
x^* = \arg\max_{x \in \mathcal{X}} f(x)
$$

where:

- $x$ = hyperparameter configuration
- $\mathcal{X}$ = search space
- $f(x)$ = validation performance
- $x^*$ = best configuration

The difficulty is that evaluating $f(x)$ can be expensive because every evaluation may require training and validating a model.

## 4. Common Hyperparameter Search Methods

### Grid Search

Tests every predefined combination.

```text
learning_rate = [0.001, 0.01, 0.1]
max_depth     = [5, 10, 20]

Total trials = 3 × 3 = 9
```

### Random Search

Randomly samples configurations from the search space.

### Bayesian Optimization

Uses previous evaluations to decide which configuration should be tested next.

> **Core idea:** Do not search blindly. Learn from previous experiments.

## 5. Bayesian Optimization — Definition

> **Bayesian Optimization is a sequential optimization technique that uses a probabilistic surrogate model and an acquisition function to efficiently optimize an expensive black-box objective function.**

For hyperparameter tuning:

> It builds a model of the relationship between hyperparameters and validation performance, then intelligently selects the next hyperparameter configuration to evaluate.

### Three key terms

1. **Objective function** — what we want to optimize.
2. **Surrogate model** — an inexpensive approximation of the objective function.
3. **Acquisition function** — chooses the next point to evaluate.

The process is:

$$
\text{Evaluate} \rightarrow \text{Learn} \rightarrow \text{Choose Next} \rightarrow \text{Evaluate} \rightarrow \cdots
$$

## 6. Intuition

Imagine we are searching for the best hyperparameter value, but we do not know the true performance curve.

We evaluate a few points:

```text
Performance
    ↑
    |                         ?
    |                     ?
    |                ●
    |          ●
    |     ●
    | ●
    +--------------------------------→ Hyperparameter
```

Bayesian Optimization uses these observations to estimate what the unknown function might look like.

It asks:

> "Given everything I have learned so far, where should I test next?"

## 7. Mathematical Foundation

Let:

$$
f(x)
$$

be the unknown objective function.

For hyperparameter tuning:

$$
x = (x_1,x_2,\ldots,x_d)
$$

could contain:

- learning rate
- maximum depth
- number of estimators
- regularization strength

We want:

$$
x^* = \arg\max_x f(x)
$$

But directly evaluating $f(x)$ is expensive.

Therefore, Bayesian Optimization constructs a surrogate model:

$$
p(f(x)\mid D)
$$

where:

$$
D = \{(x_1,y_1),(x_2,y_2),\ldots,(x_n,y_n)\}
$$

is the set of observations collected so far.

## 8. Surrogate Model

A **surrogate model** approximates the expensive objective function.

Instead of repeatedly training the real ML model everywhere, we use a cheap probabilistic model to estimate promising regions.

A classical choice is the **Gaussian Process (GP)**.

Other Bayesian optimization approaches can use tree-based probabilistic models, such as the **Tree-structured Parzen Estimator (TPE)**.

The surrogate model provides information about:

- expected performance
- uncertainty

## 9. Gaussian Process

A Gaussian Process can model an unknown function probabilistically.

For every candidate $x$, we can obtain:

$$
\mu(x)
$$

= predicted mean performance

and

$$
\sigma(x)
$$

= uncertainty about the prediction.

Conceptually:

```text
             High uncertainty
                  ↓
Performance   ┌───────────┐
      ↑       │           │
      |   ●---│---●-------│---●
      |       │           │
      +----------------------------→ x
```

Near observed points, uncertainty may be smaller.

Far from observed points, uncertainty may be larger.

This uncertainty is important because Bayesian Optimization does not only search where the prediction is high; it can also investigate uncertain regions.

## 10. Acquisition Function

The surrogate model tells us what the objective may look like.

But we still need to decide:

> **Which point should we evaluate next?**

The **acquisition function** answers this question.

Common acquisition functions:

- Expected Improvement (EI)
- Probability of Improvement (PI)
- Upper Confidence Bound (UCB)

The next point is selected by:

$$
x_{next} = \arg\max_x a(x)
$$

where $a(x)$ is the acquisition function.

## 11. Expected Improvement

Suppose the best observed performance is:

$$
f_{best}
$$

Expected Improvement asks:

$$
EI(x)=\mathbb{E}[\max(f(x)-f_{best},0)]
$$

Intuitively:

> "If I evaluate this point, how much improvement do I expect over my current best result?"

A point can have high acquisition value because:

- its predicted performance is high (**exploitation**), or
- its uncertainty is high (**exploration**).

That is the central trade-off in Bayesian Optimization.

## 12. Exploration vs Exploitation

### Exploitation

Test areas where the surrogate model predicts high performance.

```text
"We already know this region looks good.
Let's search around it."
```

### Exploration

Test areas where uncertainty is high.

```text
"We don't know much about this region.
Maybe it contains a better solution."
```

Bayesian Optimization balances:

$$
\boxed{\text{Exploration} + \text{Exploitation}}
$$

This is one of its biggest advantages over methods that ignore previous observations.

## 13. Bayesian Optimization Workflow

```text
       Define Search Space
              ↓
       Initial Evaluations
              ↓
       Build Surrogate
              ↓
    Calculate Acquisition
              ↓
    Select Next Hyperparameters
              ↓
       Train ML Model
              ↓
    Evaluate Validation Score
              ↓
       Add New Observation
              ↓
       Update Surrogate
              ↓
          Repeat
              ↓
      Return Best Parameters
```

### Mathematical loop

1. Start with $D$.
2. Fit $p(f|D)$.
3. Optimize acquisition $a(x)$.
4. Select $x_{next}$.
5. Evaluate expensive $f(x_{next})$.
6. Update:

$$
D \leftarrow D \cup \{(x_{next}, f(x_{next}))\}
$$

7. Repeat until the evaluation budget is exhausted.

## 14. Import Libraries

In [ ]:
import sys
import subprocess
import importlib.util

# Install Optuna only if it is not already available.
if importlib.util.find_spec("optuna") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import optuna

print("Optuna version:", optuna.__version__)

## 15. Load a Dataset

We will use the **Breast Cancer Wisconsin dataset** included in scikit-learn.

This keeps the notebook self-contained and reproducible.

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("Number of classes:", y.nunique())
print()
print(y.value_counts())

## 16. Train/Test Split

The test set will remain untouched until the final evaluation.

The optimizer will use cross-validation on the training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 17. Baseline Model

Before tuning, create a reasonable baseline Random Forest.

This gives us a reference point for measuring improvement.

In [ ]:
baseline_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

baseline_cv = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

baseline_model.fit(X_train, y_train)
baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_model.predict(X_test)
)

print(f"Baseline CV accuracy: {baseline_cv.mean():.4f}")
print(f"Baseline test accuracy: {baseline_test_accuracy:.4f}")

## 18. Define the Bayesian Optimization Objective

We will optimize three Random Forest hyperparameters:

- `n_estimators`
- `max_depth`
- `min_samples_split`

The objective function returns the mean 5-fold cross-validation accuracy.

> Important: the test set is not used during optimization.

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int(
        "n_estimators", 50, 400
    )

    max_depth = trial.suggest_int(
        "max_depth", 2, 30
    )

    min_samples_split = trial.suggest_int(
        "min_samples_split", 2, 20
    )

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    ).mean()

    return score

## 19. Create the Optimization Study

We want to **maximize** validation accuracy.

Optuna provides multiple samplers. Here we explicitly use its **TPE sampler**, a Bayesian-style sequential optimization approach.

```text
Search Space
     ↓
TPE proposes configuration
     ↓
Random Forest + CV
     ↓
Score
     ↓
TPE learns from the trial
     ↓
Propose next configuration
```

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=42
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=False
)

print("Optimization finished.")
print("Number of trials:", len(study.trials))

## 20. Best Hyperparameters

The optimizer has now searched the hyperparameter space and identified the best observed configuration.

In [ ]:
print("Best CV accuracy:")
print(f"{study.best_value:.4f}")

print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

## 21. Evaluate the Tuned Model on the Test Set

Now, and only now, we use the test set for final evaluation.

In [ ]:
best_model = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

tuned_test_accuracy = accuracy_score(
    y_test,
    best_model.predict(X_test)
)

print(f"Baseline test accuracy: {baseline_test_accuracy:.4f}")
print(f"Tuned test accuracy:    {tuned_test_accuracy:.4f}")
print(f"Improvement:            {tuned_test_accuracy - baseline_test_accuracy:+.4f}")

## 22. Visualize Optimization History

The optimization history shows how the best observed score changes as trials are performed.

A good optimization process often improves quickly in early trials and then gradually becomes more stable.

In [ ]:
trials = study.trials

trial_numbers = [t.number for t in trials if t.value is not None]
trial_values = [t.value for t in trials if t.value is not None]

best_so_far = np.maximum.accumulate(trial_values)

plt.figure(figsize=(10, 5))
plt.scatter(trial_numbers, trial_values, alpha=0.65, label="Trial score")
plt.plot(trial_numbers, best_so_far, linewidth=2, label="Best so far")
plt.xlabel("Trial")
plt.ylabel("5-fold CV Accuracy")
plt.title("Bayesian Optimization History")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 23. Visualize Hyperparameter Search

Let's inspect how the trials performed across the hyperparameter space.

In [ ]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "state")
)

trials_df = trials_df[trials_df["state"] == "COMPLETE"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(
    trials_df["params_n_estimators"],
    trials_df["value"],
    alpha=0.7
)
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("CV Accuracy")
axes[0].set_title("n_estimators vs Score")

axes[1].scatter(
    trials_df["params_max_depth"],
    trials_df["value"],
    alpha=0.7
)
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("CV Accuracy")
axes[1].set_title("max_depth vs Score")

axes[2].scatter(
    trials_df["params_min_samples_split"],
    trials_df["value"],
    alpha=0.7
)
axes[2].set_xlabel("min_samples_split")
axes[2].set_ylabel("CV Accuracy")
axes[2].set_title("min_samples_split vs Score")

plt.tight_layout()
plt.show()

## 24. Hyperparameter Importance

Optuna can estimate which hyperparameters had the strongest relationship with the objective value in the observed trials.

This is useful for understanding the search space.

In [ ]:
try:
    importances = optuna.importance.get_param_importances(study)

    importance_df = pd.Series(importances).sort_values()

    plt.figure(figsize=(8, 4))
    importance_df.plot(kind="barh")
    plt.xlabel("Importance")
    plt.ylabel("Hyperparameter")
    plt.title("Hyperparameter Importance")
    plt.grid(axis="x", alpha=0.25)
    plt.show()

    print(importance_df.sort_values(ascending=False))
except Exception as e:
    print("Importance visualization could not be generated:", e)

## 25. Best Trials

Let's inspect the top configurations discovered by the optimizer.

In [ ]:
top_trials = (
    trials_df
    .sort_values("value", ascending=False)
    .head(10)
)

display(top_trials)

## 26. Manual Bayesian Optimization Concept

The following example is **not** a replacement for a full Bayesian optimization library. It is a small educational simulation showing the central idea.

We create an unknown objective function, observe a few points, fit a Gaussian Process, and use a simple Upper Confidence Bound acquisition rule.

This makes the mathematical workflow visible.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

rng = np.random.default_rng(42)

# Unknown function that we pretend is expensive.
def hidden_function(x):
    return (
        np.sin(3 * x)
        + 0.5 * np.cos(7 * x)
        + 0.15 * x
    )

# Initial observations.
X_obs = np.array([[0.1], [0.4], [0.8], [1.2], [1.7]])
y_obs = hidden_function(X_obs.ravel())

# Candidate search space.
X_grid = np.linspace(0, 2, 500).reshape(-1, 1)

kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(length_scale=0.4, nu=2.5)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.05**2,
    normalize_y=True,
    random_state=42
)

gp.fit(X_obs, y_obs)

mean, std = gp.predict(
    X_grid,
    return_std=True
)

# Upper Confidence Bound.
beta = 2.0
ucb = mean + beta * std

next_index = np.argmax(ucb)
x_next = X_grid[next_index, 0]
y_next = hidden_function(x_next)

print(f"Suggested next x: {x_next:.4f}")
print(f"Observed objective at x: {y_next:.4f}")

## 27. Visualizing the Surrogate Model

The plot below shows:

- observed points
- predicted mean
- uncertainty band
- UCB acquisition function
- suggested next point

This is the basic visual intuition behind Gaussian-Process Bayesian Optimization.

In [ ]:
plt.figure(figsize=(11, 6))

plt.plot(
    X_grid.ravel(),
    hidden_function(X_grid.ravel()),
    linestyle="--",
    linewidth=1.5,
    label="Unknown true function"
)

plt.plot(
    X_grid.ravel(),
    mean,
    linewidth=2,
    label="GP predicted mean"
)

plt.fill_between(
    X_grid.ravel(),
    mean - 1.96 * std,
    mean + 1.96 * std,
    alpha=0.20,
    label="Approx. 95% uncertainty"
)

plt.plot(
    X_grid.ravel(),
    ucb,
    linestyle=":",
    linewidth=2,
    label="UCB acquisition"
)

plt.scatter(
    X_obs.ravel(),
    y_obs,
    s=60,
    label="Observed points",
    zorder=5
)

plt.scatter(
    [x_next],
    [y_next],
    s=100,
    marker="*",
    label="Suggested next point",
    zorder=6
)

plt.xlabel("Hyperparameter / Search Variable")
plt.ylabel("Objective Value")
plt.title("Bayesian Optimization: Surrogate + Acquisition")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 28. What Happened in the Manual Example?

The process was:

### Step 1
We observed a few values of the unknown function.

### Step 2
The Gaussian Process estimated:

$$
\mu(x), \sigma(x)
$$

for the entire search space.

### Step 3
We created the UCB acquisition function:

$$
UCB(x) = \mu(x) + \beta\sigma(x)
$$

### Step 4
We selected:

$$
x_{next} = \arg\max_x UCB(x)
$$

### Step 5
We evaluated the expensive/unknown function at that point.

This is the same fundamental loop used by Bayesian Optimization systems, although production libraries may use different surrogate models and acquisition/sampling strategies.

## 29. Grid Search vs Random Search vs Bayesian Optimization

| Property | Grid Search | Random Search | Bayesian Optimization |
|---|---|---|---|
| Uses previous results | No | No | Yes |
| Surrogate model | No | No | Yes |
| Acquisition / sequential decision | No | No | Yes |
| Search efficiency | Low for large spaces | Good | Often very good |
| Expensive objective | Inefficient | Better | Strong choice |
| Parallelization | Easy | Easy | More challenging |
| Simplicity | Very high | High | Moderate |
| Best use case | Small search space | Large/simple spaces | Expensive evaluations |

### Main difference

**Grid Search:**

> "Try everything."

**Random Search:**

> "Try random things."

**Bayesian Optimization:**

> "Use what I learned to decide what to try next."

## 30. Advantages of Bayesian Optimization

### 1. Efficient
It can find good configurations using relatively few evaluations.

### 2. Learns from previous trials
Each evaluation provides information for later decisions.

### 3. Good for expensive models
Especially useful when one training run takes minutes or hours.

### 4. Handles complex search spaces
It can optimize multiple hyperparameters together.

### 5. Balances exploration and exploitation
It can investigate uncertain regions instead of only exploiting the currently best region.

## 31. Disadvantages

### 1. Sequential nature
Later trials depend on earlier observations, which can limit parallelism.

### 2. More complex
It is conceptually and computationally more involved than Grid Search.

### 3. Overhead
Building and updating the optimization model has computational cost.

### 4. Not always necessary
For very cheap models, Random Search or Grid Search may be simpler and sufficiently effective.

### 5. Performance depends on the optimization setup
Search spaces, sampling method, objective noise, and trial budget all matter.

## 32. When Should You Use Bayesian Optimization?

Use it when:

- model training is expensive
- the number of experiments is limited
- the hyperparameter space is reasonably complex
- you want to make every trial informative
- you have a costly black-box objective

Examples:

```text
Deep Learning
XGBoost
LightGBM
Random Forest
SVM
Neural network architecture/search
```

For a model that trains in milliseconds, the extra optimization machinery may not be worth it.

## 33. Practical Tips

### Use a validation strategy
Do not optimize on the test set.

### Use cross-validation when appropriate
For small/medium datasets, CV can provide a more reliable objective.

### Set a sensible search space
A poor search space can make any optimizer perform poorly.

### Use a reproducible seed
This makes experiments easier to compare.

### Give the optimizer enough trials
Too few trials can prevent it from discovering good regions.

### Consider logarithmic ranges
For parameters such as learning rate, use log-scaled suggestions when appropriate.

Example:

```python
trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
```

## 34. Important Distinction: Bayesian Optimization vs Optuna

Do not memorize:

> "Optuna is Bayesian Optimization."

That statement is incomplete.

**Optuna** is a hyperparameter optimization framework that provides multiple samplers.

One of its commonly used samplers is **TPE (Tree-structured Parzen Estimator)**.

Therefore:

> **Optuna is a framework; TPE is one optimization/sampling approach provided by Optuna.**

Similarly, Bayesian Optimization is a broader family of sequential optimization techniques.

## 35. Exam-Ready Answer

### Definition

**Bayesian Optimization is a sequential optimization technique that efficiently finds optimal hyperparameters by modeling an expensive objective function using a probabilistic surrogate model and selecting new configurations using an acquisition function.**

### Main components

1. **Objective function** — measures model performance.
2. **Surrogate model** — approximates the expensive objective.
3. **Acquisition function** — chooses the next configuration.
4. **Exploration** — searches uncertain regions.
5. **Exploitation** — searches promising regions.

### Mathematical form

$$
x^* = \arg\max_x f(x)
$$

Surrogate:

$$
p(f(x)\mid D)
$$

Next point:

$$
x_{next}=\arg\max_x a(x)
$$

Update:

$$
D \leftarrow D \cup \{(x_{next},f(x_{next}))\}
$$

## 36. One-Minute Mental Model

Remember Bayesian Optimization as:

```text
         Previous Trials
               ↓
       ┌───────────────┐
       │ Surrogate     │
       │ Model         │
       └───────┬───────┘
               ↓
      Acquisition Function
               ↓
      Best Next Hyperparameters
               ↓
          Train Model
               ↓
        Get Validation Score
               ↓
        Add New Observation
               │
               └──────────────→ Repeat
```

### The three words to remember:

$$
\boxed{\text{Surrogate + Acquisition + Sequential}}
$$

### The two ideas to remember:

$$
\boxed{\text{Exploration + Exploitation}}
$$

## 37. Final Takeaways

- Hyperparameter tuning searches for good configurations before final model deployment.
- Grid Search exhaustively tests a predefined grid.
- Random Search samples configurations randomly.
- Bayesian Optimization learns from previous evaluations.
- A surrogate model approximates the expensive objective function.
- Gaussian Processes provide a predicted mean and uncertainty.
- An acquisition function decides which point should be evaluated next.
- Expected Improvement is a popular acquisition strategy.
- Bayesian Optimization balances exploration and exploitation.
- It is especially valuable when model evaluations are expensive.
- Optuna is a practical framework for hyperparameter optimization; its TPE sampler is one Bayesian-style approach.

## 38. Quick Quiz

Try answering these without looking back.

1. What is a hyperparameter?
2. What is the main problem with Grid Search?
3. What does the surrogate model do?
4. What is an acquisition function?
5. What is Expected Improvement?
6. What is exploitation?
7. What is exploration?
8. Why is Bayesian Optimization useful for expensive models?
9. What is the mathematical objective of hyperparameter tuning?
10. Is Optuna itself a single Bayesian Optimization algorithm?

### Answers

1. A hyperparameter is a configuration value selected outside the normal parameter-learning process.
2. It can require many expensive model evaluations.
3. It approximates the expensive objective function.
4. It selects promising candidates for the next evaluation.
5. EI estimates the expected improvement over the current best result.
6. Exploitation searches regions already predicted to perform well.
7. Exploration investigates uncertain regions.
8. It uses previous evaluations to reduce wasted experiments.
9. $x^*=\arg\max_x f(x)$ for maximization.
10. No. Optuna is a framework with multiple optimization/sampling methods, including TPE.

# End of Notebook

**Core formula:**

$$
\boxed{x^* = \arg\max_x f(x)}
$$

**Core workflow:**

$$
\boxed{
\text{Evaluate}
\rightarrow
\text{Surrogate}
\rightarrow
\text{Acquisition}
\rightarrow
\text{Next Trial}
\rightarrow
\text{Repeat}
}
$$

**Core concept:**

$$
\boxed{\text{Exploration + Exploitation}}
$$